In [40]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

from sklearn.neural_network import MLPRegressor

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.metrics import mean_absolute_error
from sklearn.decomposition import PCA

from xgboost import XGBRegressor

In [41]:
df = pd.read_csv('gurgaon_properties_missing_value_imputation.csv')

In [42]:
df.head()

,property_type,society,sector,price,price_per_sqft,bedRoom,bathroom,balcony,floorNum,agePossession,built_up_area,study room,servant room,store room,pooja room,others,furnishing_type,luxury_score
0,flat,signature global park 4,sector 36,0.82,7586.0,3.0,2.0,2,2.0,New Property,850.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
1,flat,smart world gems,sector 89,0.95,8597.0,2.0,2.0,2,4.0,New Property,1226.0,1.0,1.0,0.0,0.0,0.0,0.0,38.0
2,flat,breez global hill view,sohna road,0.32,5470.0,2.0,2.0,1,17.0,New Property,1000.0,0.0,0.0,0.0,0.0,0.0,0.0,49.0
3,flat,bestech park view sanskruti,sector 92,1.60,8020.0,3.0,4.0,3+,10.0,Relatively New,1615.0,0.0,1.0,0.0,0.0,1.0,1.0,174.0
4,flat,suncity avenue,sector 102,0.48,9023.0,2.0,2.0,1,5.0,Relatively New,582.0,0.0,0.0,1.0,0.0,0.0,0.0,159.0


In [43]:
df['furnishing_type'].value_counts()

furnishing_type
0.0    2349
1.0    1018
2.0     187
Name: count, dtype: int64

In [44]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished
df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [45]:
df.head()

,property_type,society,sector,price,price_per_sqft,bedRoom,bathroom,balcony,floorNum,agePossession,built_up_area,study room,servant room,store room,pooja room,others,furnishing_type,luxury_score
0,flat,signature global park 4,sector 36,0.82,7586.0,3.0,2.0,2,2.0,New Property,850.0,0.0,0.0,0.0,0.0,0.0,unfurnished,8.0
1,flat,smart world gems,sector 89,0.95,8597.0,2.0,2.0,2,4.0,New Property,1226.0,1.0,1.0,0.0,0.0,0.0,unfurnished,38.0
2,flat,breez global hill view,sohna road,0.32,5470.0,2.0,2.0,1,17.0,New Property,1000.0,0.0,0.0,0.0,0.0,0.0,unfurnished,49.0
3,flat,bestech park view sanskruti,sector 92,1.60,8020.0,3.0,4.0,3+,10.0,Relatively New,1615.0,0.0,1.0,0.0,0.0,1.0,semifurnished,174.0
4,flat,suncity avenue,sector 102,0.48,9023.0,2.0,2.0,1,5.0,Relatively New,582.0,0.0,0.0,1.0,0.0,0.0,unfurnished,159.0


In [46]:
X = df.drop(columns=['price', 'society', 'price_per_sqft'])
y = df['price']

In [47]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

### Ordinal Encoding

In [48]:
# columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [49]:
# # Creating a column transformer for preprocessing
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
#         ('cat', OrdinalEncoder(), columns_to_encode)
#     ], 
#     remainder='passthrough'
# )

In [50]:
# # Creating a pipeline
# pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('regressor', LinearRegression())
# ])

In [51]:
X = df.drop(columns=['price', 'society', 'price_per_sqft'])
y = df['price']
y_transformed = np.log1p(y)

columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 
                                   'servant room', 'store room', 'luxury_score',
                                   'floorNum', 'study room', 'pooja room', 'others']),
        ('cat', OrdinalEncoder(), columns_to_encode)
    ], 
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

print(scores.mean(), scores.std())

0.7403576346914912 0.0327107362005436


In [52]:
print(X.columns.tolist())


['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony', 'floorNum', 'agePossession', 'built_up_area', 'study room', 'servant room', 'store room', 'pooja room', 'others', 'furnishing_type', 'luxury_score']


In [53]:
scores.mean(),scores.std()

(np.float64(0.7403576346914912), np.float64(0.0327107362005436))

In [54]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [55]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3554 non-null   str    
 1   society          3554 non-null   str    
 2   sector           3554 non-null   str    
 3   price            3554 non-null   float64
 4   price_per_sqft   3554 non-null   float64
 5   bedRoom          3554 non-null   float64
 6   bathroom         3554 non-null   float64
 7   balcony          3554 non-null   str    
 8   floorNum         3554 non-null   float64
 9   agePossession    3554 non-null   str    
 10  built_up_area    3554 non-null   float64
 11  study room       3554 non-null   float64
 12  servant room     3554 non-null   float64
 13  store room       3554 non-null   float64
 14  pooja room       3554 non-null   float64
 15  others           3554 non-null   float64
 16  furnishing_type  3554 non-null   object 
 17  luxury_score     3554 non

In [56]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [57]:
y_pred = pipeline.predict(X_test)

In [58]:
y_pred = np.expm1(y_pred)

In [59]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.9359102048516716

In [60]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [61]:
model_dict = {
    'linear_reg': LinearRegression(),
    'svr': SVR(),
    'ridge': Ridge(),
    'lasso': Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest': RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost': XGBRegressor()
}

In [62]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [63]:
model_output

[['linear_reg', np.float64(0.7403576346914912), 0.9359102048516716],
 ['svr', np.float64(0.7670703966136901), 0.8349074797757188],
 ['ridge', np.float64(0.7403614351826855), 0.9358335983462466],
 ['lasso', np.float64(0.05943378064493573), 1.528905986892753],
 ['decision tree', np.float64(0.7771323460379318), 0.6924049926736481],
 ['random forest', np.float64(0.881680578425369), 0.5364918363350138],
 ['extra trees', np.float64(0.8716947629379549), 0.53473826753112],
 ['gradient boosting', np.float64(0.8746169344884027), 0.5654469871097109],
 ['adaboost', np.float64(0.7536653804579871), 0.8421729501596337],
 ['mlp', np.float64(0.8115648548595461), 0.6718398703353379],
 ['xgboost', np.float64(0.8915130816037646), 0.5002182765592167]]

In [64]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [65]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.891513,0.500218
6,extra trees,0.871695,0.534738
5,random forest,0.881681,0.536492
7,gradient boosting,0.874617,0.565447
9,mlp,0.811565,0.671840
4,decision tree,0.777132,0.692405
1,svr,0.767070,0.834907
8,adaboost,0.753665,0.842173
2,ridge,0.740361,0.935834
0,linear_reg,0.740358,0.935910


### OneHotEncoding

In [66]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first'),['sector','agePossession','furnishing_type'])
    ], 
    remainder='passthrough'
)

In [67]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [68]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [69]:
scores.mean()

np.float64(0.8556939314028928)

In [70]:
scores.std()

np.float64(0.015795103024873542)

In [71]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [72]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [73]:
y_pred = pipeline.predict(X_test)

In [74]:
y_pred = np.expm1(y_pred)

In [75]:
df['luxury_category'] = pd.qcut(
    df['luxury_score'],
    q=4,
    labels=['Basic', 'Standard', 'Premium', 'Ultra Luxury']
)

In [76]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.6485145931438183

In [77]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [78]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [79]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [80]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [81]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.897509,0.457165
10,xgboost,0.895962,0.488562
5,random forest,0.891484,0.496605
7,gradient boosting,0.877101,0.566494
9,mlp,0.848106,0.570656
0,linear_reg,0.855694,0.648515
2,ridge,0.855589,0.652276
4,decision tree,0.801209,0.718340
8,adaboost,0.755611,0.843475
1,svr,0.684170,0.917890


### OneHotEncoding With PCA

In [82]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['sector','agePossession'])
    ], 
    remainder='passthrough'
)

In [83]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [84]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [85]:
scores.mean()

np.float64(0.09139572153578901)

In [86]:
scores.std()

np.float64(0.019524503315560594)

In [87]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [88]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [89]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [90]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [91]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.487923,1.043959
5,random forest,0.487491,1.057287
10,xgboost,0.490850,1.064830
7,gradient boosting,0.432717,1.184358
1,svr,0.239226,1.363750
4,decision tree,0.202003,1.420829
9,mlp,0.221257,1.428458
8,adaboost,0.256978,1.435820
2,ridge,0.091396,1.508013
0,linear_reg,0.091396,1.508013


### Target Encoder

In [92]:
!pip install category_encoders


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [93]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [94]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [97]:
X = df.drop(columns=['price', 'society', 'price_per_sqft'])
y = df['price']
y_transformed = np.log1p(y)

columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 
                                   'servant room', 'store room',
                                   'floorNum', 'study room', 'pooja room', 'others']),  # 👈 removed luxury_category
        ('cat', OrdinalEncoder(), ['property_type', 'sector', 'balcony', 'agePossession', 
                                   'furnishing_type', 'luxury_category'])  # 👈 added here
    ], 
    remainder='drop'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

print(scores.mean(), scores.std())

0.740697203328981 0.03189855371512419


In [98]:
print("X Columns:")
for col in X.columns:
    print(repr(col))

X Columns:
'property_type'
'sector'
'bedRoom'
'bathroom'
'balcony'
'floorNum'
'agePossession'
'built_up_area'
'study room'
'servant room'
'store room'
'pooja room'
'others'
'furnishing_type'
'luxury_score'
'luxury_category'


In [126]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   property_type    3554 non-null   str     
 1   society          3554 non-null   str     
 2   sector           3554 non-null   str     
 3   price            3554 non-null   float64 
 4   price_per_sqft   3554 non-null   float64 
 5   bedRoom          3554 non-null   float64 
 6   bathroom         3554 non-null   float64 
 7   balcony          3554 non-null   str     
 8   floorNum         3554 non-null   float64 
 9   agePossession    3554 non-null   str     
 10  built_up_area    3554 non-null   float64 
 11  study room       3554 non-null   float64 
 12  servant room     3554 non-null   float64 
 13  store room       3554 non-null   float64 
 14  pooja room       3554 non-null   float64 
 15  others           3554 non-null   float64 
 16  furnishing_type  3554 non-null   object  
 17  luxury

In [ ]:
# Removed: search.estimator referenced before search.fit() is called

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat1',
                                                  OneHo

In [99]:
pipeline.fit(X, y_transformed)
print("Pipeline fit successful")

Pipeline fit successful


In [127]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   property_type    3554 non-null   str     
 1   society          3554 non-null   str     
 2   sector           3554 non-null   str     
 3   price            3554 non-null   float64 
 4   price_per_sqft   3554 non-null   float64 
 5   bedRoom          3554 non-null   float64 
 6   bathroom         3554 non-null   float64 
 7   balcony          3554 non-null   str     
 8   floorNum         3554 non-null   float64 
 9   agePossession    3554 non-null   str     
 10  built_up_area    3554 non-null   float64 
 11  study room       3554 non-null   float64 
 12  servant room     3554 non-null   float64 
 13  store room       3554 non-null   float64 
 14  pooja room       3554 non-null   float64 
 15  others           3554 non-null   float64 
 16  furnishing_type  3554 non-null   object  
 17  luxury

In [128]:
df = df.drop(columns=['luxury_score'])

In [129]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   property_type    3554 non-null   str     
 1   society          3554 non-null   str     
 2   sector           3554 non-null   str     
 3   price            3554 non-null   float64 
 4   price_per_sqft   3554 non-null   float64 
 5   bedRoom          3554 non-null   float64 
 6   bathroom         3554 non-null   float64 
 7   balcony          3554 non-null   str     
 8   floorNum         3554 non-null   float64 
 9   agePossession    3554 non-null   str     
 10  built_up_area    3554 non-null   float64 
 11  study room       3554 non-null   float64 
 12  servant room     3554 non-null   float64 
 13  store room       3554 non-null   float64 
 14  pooja room       3554 non-null   float64 
 15  others           3554 non-null   float64 
 16  furnishing_type  3554 non-null   object  
 17  luxury

In [100]:
scores.mean(),scores.std()

(np.float64(0.740697203328981), np.float64(0.03189855371512419))

In [101]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [102]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [103]:
model_output = []
for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [104]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [105]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.889860,0.501941
5,random forest,0.882662,0.533992
6,extra trees,0.872042,0.544688
7,gradient boosting,0.874654,0.563057
4,decision tree,0.778670,0.695794
9,mlp,0.813068,0.726708
1,svr,0.767332,0.834386
8,adaboost,0.750590,0.842514
2,ridge,0.740701,0.933532
0,linear_reg,0.740697,0.933585


### Hyperparameter Tuning

In [106]:
from sklearn.model_selection import GridSearchCV

In [130]:
param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['sqrt', 1.0]
}

In [131]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   property_type    3554 non-null   str     
 1   society          3554 non-null   str     
 2   sector           3554 non-null   str     
 3   price            3554 non-null   float64 
 4   price_per_sqft   3554 non-null   float64 
 5   bedRoom          3554 non-null   float64 
 6   bathroom         3554 non-null   float64 
 7   balcony          3554 non-null   str     
 8   floorNum         3554 non-null   float64 
 9   agePossession    3554 non-null   str     
 10  built_up_area    3554 non-null   float64 
 11  study room       3554 non-null   float64 
 12  servant room     3554 non-null   float64 
 13  store room       3554 non-null   float64 
 14  pooja room       3554 non-null   float64 
 15  others           3554 non-null   float64 
 16  furnishing_type  3554 non-null   object  
 17  luxury

In [133]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   property_type    3554 non-null   str     
 1   society          3554 non-null   str     
 2   sector           3554 non-null   str     
 3   price            3554 non-null   float64 
 4   price_per_sqft   3554 non-null   float64 
 5   bedRoom          3554 non-null   float64 
 6   bathroom         3554 non-null   float64 
 7   balcony          3554 non-null   str     
 8   floorNum         3554 non-null   float64 
 9   agePossession    3554 non-null   str     
 10  built_up_area    3554 non-null   float64 
 11  study room       3554 non-null   float64 
 12  servant room     3554 non-null   float64 
 13  store room       3554 non-null   float64 
 14  pooja room       3554 non-null   float64 
 15  others           3554 non-null   float64 
 16  furnishing_type  3554 non-null   object  
 17  luxury

In [134]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   property_type    3554 non-null   str     
 1   society          3554 non-null   str     
 2   sector           3554 non-null   str     
 3   price            3554 non-null   float64 
 4   price_per_sqft   3554 non-null   float64 
 5   bedRoom          3554 non-null   float64 
 6   bathroom         3554 non-null   float64 
 7   balcony          3554 non-null   str     
 8   floorNum         3554 non-null   float64 
 9   agePossession    3554 non-null   str     
 10  built_up_area    3554 non-null   float64 
 11  study room       3554 non-null   float64 
 12  servant room     3554 non-null   float64 
 13  store room       3554 non-null   float64 
 14  pooja room       3554 non-null   float64 
 15  others           3554 non-null   float64 
 16  furnishing_type  3554 non-null   object  
 17  luxury

In [135]:
# (removed empty cell)

In [136]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   property_type    3554 non-null   str     
 1   society          3554 non-null   str     
 2   sector           3554 non-null   str     
 3   price            3554 non-null   float64 
 4   price_per_sqft   3554 non-null   float64 
 5   bedRoom          3554 non-null   float64 
 6   bathroom         3554 non-null   float64 
 7   balcony          3554 non-null   str     
 8   floorNum         3554 non-null   float64 
 9   agePossession    3554 non-null   str     
 10  built_up_area    3554 non-null   float64 
 11  study room       3554 non-null   float64 
 12  servant room     3554 non-null   float64 
 13  store room       3554 non-null   float64 
 14  pooja room       3554 non-null   float64 
 15  others           3554 non-null   float64 
 16  furnishing_type  3554 non-null   object  
 17  luxury

In [137]:
columns_to_encode = ['property_type', 'balcony', 'furnishing_type', 'luxury_category', 'floor_category']  
# 👆 removed 'agePossession' and 'sector' — handled separately below

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), [
            'bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room'
        ]),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1', OneHotEncoder(drop='first', sparse_output=False), ['agePossession']),  # 👈 only here
        ('target_enc', ce.TargetEncoder(), ['sector'])  # 👈 only here
    ], 
    remainder='drop'  # 👈 safer than passthrough
)

In [138]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

In [139]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [140]:
search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4)

In [141]:
search.fit(X, y_transformed)

Fitting 10 folds for each of 128 candidates, totalling 1280 fits


ValueError: 
All the 1280 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1280 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\indexes\base.py", line 3641, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 168, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 197, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7668, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7676, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'floor_categor

In [142]:
all_handled = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room',
               'property_type', 'balcony', 'furnishing_type', 'luxury_category', 
               'floor_category', 'agePossession', 'sector']

remaining = [c for c in X.columns if c not in all_handled]
print(remaining)  # these will be passed through as-is — check if any are strings

['floorNum', 'study room', 'pooja room', 'others', 'luxury_score']


In [143]:
final_pipe = search.best_estimator_

AttributeError: 'GridSearchCV' object has no attribute 'best_estimator_'

In [144]:
search.best_params_

AttributeError: 'GridSearchCV' object has no attribute 'best_params_'

In [145]:
search.best_score_

AttributeError: 'GridSearchCV' object has no attribute 'best_score_'

In [146]:
final_pipe.fit(X,y_transformed)

NameError: name 'final_pipe' is not defined

### Exporting the model

In [147]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['sector','agePossession'])
    ], 
    remainder='passthrough'
)

In [148]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=500))
])

In [149]:
pipeline.fit(X,y_transformed)

ValueError: A given column is not a column of the dataframe

In [150]:
import pickle

with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

In [151]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

In [ ]:
X

### Trying out the predictions

In [ ]:
X.columns

In [ ]:
X.iloc[0].values

In [ ]:
data = [['house', 'sector 102', 4, 3, '3+', 'New Property', 2750, 0, 0, 'unfurnished', 'Low', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category']

# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df


In [ ]:
np.expm1(pipeline.predict(one_df))

In [ ]:
X.dtypes

In [ ]:
sorted(X['sector'].unique().tolist())

In [ ]:
# (removed empty cell)

In [ ]:
pipeline.fit(X, y_transformed)

import pickle

# Save trained pipeline
with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

# Save dataframe used for training
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)